# schedule_raw_re_opp_custom_fields
Fetches `/opportunity/v1/opportunities/{id}/customfields` for every opportunity ID
in the `raw_re_opportunities` Domo dataset.

In [ ]:
%run user_configuration.ipynb
%run renxt_core.ipynb


In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
DOMO_INPUT_DATASET  = "raw_re_opportunities"            # source of opportunity IDs
ID_COLUMN           = "id"
DOMO_OUTPUT_DATASET = "raw_re_opportunity_custom_fields" # replace with UUID
OPP_CF_URL_TPL      = API_BASE + "/opportunity/v1/opportunities/{opp_id}/customfields"
# ─────────────────────────────────────────────────────────────────────────────


In [ ]:
df_opps = domo.read_dataframe(DOMO_INPUT_DATASET, query="SELECT * FROM table")
opp_ids = df_opps[ID_COLUMN].dropna().astype(str).str.strip().unique()
print(f"Loaded {len(opp_ids):,} opportunity IDs")


In [ ]:
token_mgr = TokenManager(interactive=False)
sess      = requests.Session()
results   = []

for i, oid in enumerate(opp_ids, 1):
    resp = api_request_with_auth(
        "GET", OPP_CF_URL_TPL.format(opp_id=oid),
        token_mgr=token_mgr, session=sess,
    )
    results.append({"opportunity_id": oid, "_status": resp.status_code,
                    "value": resp.json() if resp.ok else None})
    if i % 500 == 0:
        print(f"  {i:,}/{len(opp_ids):,} fetched")

print(f"Done. {sum(r['_status']==200 for r in results):,}/{len(results):,} succeeded.")


In [ ]:
rows = []
for r in results:
    if r["_status"] != 200 or not r["value"]:
        continue
    val   = r["value"]
    items = val if isinstance(val, list) else (val.get("value") or [val])
    for item in items:
        if isinstance(item, dict):
            item["opportunity_id"] = r["opportunity_id"]
            rows.append(item)

df_opp_cf = pd.DataFrame(rows) if rows else pd.DataFrame(columns=["opportunity_id"])
print(f"Flattened to {len(df_opp_cf):,} rows, {len(df_opp_cf.columns)} columns")


In [ ]:
df_out = domo_safe_cast(df_opp_cf)
domo.write_dataframe(df_out, dataset=DOMO_OUTPUT_DATASET, update_method="REPLACE")
print(f"✅ Wrote {len(df_out):,} rows → {DOMO_OUTPUT_DATASET}")
